# 06 - Check the local API

Serve the saved bundle through FastAPI. Check readiness, valid input, invalid input, batch size, and uncertain results. These tests use an in-process client; they are not network latency measurements. Complete the setup for 05 first.

In [1]:
from pathlib import Path
import json
import platform
import sys

assert platform.system() == "Linux", "Use the WSL kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib.util
assert importlib.util.find_spec("fastapi"), "Run setup_local.sh and restart the kernel."
from fastapi.testclient import TestClient
from service.app import create_app


/home/dew_perawis/.venvs/mlop-pj/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Run the checks

The model loads once at startup. A bad model blocks readiness with 503; invalid input returns 422. Stop on a failed check. API dependencies must be installed so API tests are not skipped.

The saved run passed all 15 tests. Its missing-file traceback came from the intentional missing-model test. The updated test captures and checks that error. Earlier outputs are kept below. To refresh them, rerun only this check cell after the setup cell if the kernel was restarted.

The TestClient warning about `httpx` is a deprecation warning, not a failed test.


In [2]:
import subprocess
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                        cwd=ROOT, capture_output=True, text=True,
                        env={**__import__("os").environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"})
print(result.stdout + result.stderr)
(ROOT / "reports/api-tests.txt").write_text(result.stdout + result.stderr)
assert result.returncode == 0 and "skipped" not in result.stderr.lower(), "Review failed or skipped checks."


/home/dew_perawis/.venvs/mlop-pj/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
test_bad_requests_are_422 (test_api.ApiTests.test_bad_requests_are_422) ... {"request_id": "4d8c4042-b41e-41c1-8b0e-7772c3e92728", "path": "/predict", "status": 422, "duration_ms": 21.871682001801673, "model_version": "abfd88e893ca"}
{"request_id": "68ef6539-613d-4c95-b810-e139407282db", "path": "/predict", "status": 422, "duration_ms": 8.515902001818176, "model_version": "abfd88e893ca"}
{"request_id": "d34faa2d-23c6-4e5c-9d6f-581a6d006961", "path": "/predict", "status": 422, "duration_ms": 3.4271240001544356, "model_version": "abfd88e893ca"}
{"request_id": "f2e5e49f-2f2f-49b4-9e63-48758d73522a", "path": "/predict", "status": 422, "duration_ms": 2.5936319943866692, "model_version": "abfd88e893ca"}
{"request_id": "168deff0-3

## 2. Send example requests

`/health` checks the process. `/ready` checks the loaded bundle. `/predict` takes 24 named ratings per profile, with at most 100 profiles. An uncertain result has status 200 and no segment recommendation.

In [3]:
fixture = json.loads((ROOT / "reports/golden-profiles.json").read_text())
with TestClient(create_app(ROOT)) as client:
    health = client.get("/health")
    ready = client.get("/ready")
    prediction = client.post("/predict", json={"profiles": fixture["profiles"]})
    bad = client.post("/predict", json={"profiles": [{"Category 1": 2}]})
    assert [health.status_code, ready.status_code, prediction.status_code, bad.status_code] == [200, 200, 200, 422]
    report = {"health": health.json(), "ready": ready.json(), "prediction": prediction.json(),
              "invalid_input_status": bad.status_code, "transport": "in-process TestClient",
              "public_deployment": False}
(ROOT / "reports/api-smoke.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))


{"request_id": "0409671b-bbc4-4da4-8f85-c27852d7687f", "path": "/health", "status": 200, "duration_ms": 22.083065996412188, "model_version": "abfd88e893ca"}
{"request_id": "8a254349-03ca-42db-bda4-63467bc7791c", "path": "/ready", "status": 200, "duration_ms": 6.534118001582101, "model_version": "abfd88e893ca"}
{"request_id": "145df6f1-361c-4994-b6db-75feba81efb6", "path": "/predict", "status": 200, "duration_ms": 29.49375699972734, "model_version": "abfd88e893ca"}
{"request_id": "a1538ba6-3ccd-4460-9468-b6db0d65d884", "path": "/predict", "status": 422, "duration_ms": 2.927651999925729, "model_version": "abfd88e893ca"}


{
  "health": {
    "status": "alive"
  },
  "ready": {
    "status": "ready",
    "model_version": "abfd88e893ca",
    "mapping_status": "provisional; review the cluster profiles"
  },
  "prediction": {
    "request_id": "145df6f1-361c-4994-b6db-75feba81efb6",
    "model_version": "abfd88e893ca",
    "mapping_status": "provisional; review the cluster profiles",
    "predictions": [
      {
        "cluster_id": 0,
        "segment": "Segment 1",
        "distance": 4.143480490277737,
        "uncertain": false,
        "model_version": "abfd88e893ca",
        "status": "ok"
      },
      {
        "cluster_id": 1,
        "segment": "Segment 2",
        "distance": 3.6429553680014135,
        "uncertain": false,
        "model_version": "abfd88e893ca",
        "status": "ok"
      },
      {
        "cluster_id": 2,
        "segment": "Segment 3",
        "distance": 4.0918361879289185,
        "uncertain": false,
        "model_version": "abfd88e893ca",
        "status": "ok"
      

## Output

`reports/api-tests.txt` and `reports/api-smoke.json`. Expect all tests to pass with no skips, status codes 200/200/200/422, and the saved model version. Segment names are still provisional. Next: real HTTP load testing, Docker, and Azure readiness.